In [2]:
"""
Module 2 — Seed Selection
==========================
Selects 8 spatially uniform seed points per library (24 total)
using area number as a proxy for spatial position on the wafer.

Strategy — Uniform Quantile Sampling:
    Sort areas within each library, divide into 8 equal quantile
    bins, pick the area closest to the centre of each bin.
    This guarantees the seed points are spread across the full
    area range (1-342) rather than clustering at one end.

Input:
    - unified_data.csv   (output of Module 1)

Output:
    - seed_set.csv       (24 rows — the initial GP training set)
    - pool_set.csv       (remaining rows — ground truth for evaluation)
"""

import numpy as np
import pandas as pd
from pathlib import Path
print("libraries imported successfully")

libraries imported successfully


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# SEED SELECTION
# ─────────────────────────────────────────────────────────────────────────────

def select_seeds_per_library(
    df          : pd.DataFrame,
    n_seeds     : int = 8,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Select n_seeds spatially uniform points from each library.

    Method — Uniform Quantile Sampling:
    ------------------------------------
    1. Sort all areas within the library (1 → 342, with gaps for excluded)
    2. Divide into n_seeds equal quantile bins
       e.g. for 322 areas and 8 seeds:
            bin 1 = areas ranked  1-40
            bin 2 = areas ranked 41-80  ... etc
    3. From each bin, pick the area closest to the bin centre
       (deterministic — same result every run, no randomness needed)

    Why this works as a spatial proxy:
    ------------------------------------
    The wafer is measured in a fixed grid pattern, so area numbers
    follow a consistent spatial order across the wafer surface.
    Uniform quantile sampling in area number ≈ uniform spatial coverage.

    Parameters
    ----------
    df           : unified DataFrame (output of Module 1)
    n_seeds      : number of seed points per library (default 8)
    random_state : kept for API consistency, not used here

    Returns
    -------
    DataFrame of selected seed points (n_seeds × n_libraries rows)
    """
    seed_frames = []

    for library, group in df.groupby('library'):
        # Sort by area number — this is our spatial proxy
        group_sorted = group.sort_values('area').reset_index(drop=True)
        n_total = len(group_sorted)

        # Divide into n_seeds equal-width bins
        # e.g. 322 areas / 8 bins = ~40 areas per bin
        bin_size = n_total / n_seeds

        selected_indices = []
        for i in range(n_seeds):
            # Centre of this bin in index space
            bin_centre = (i + 0.5) * bin_size

            # Pick the row whose index is closest to bin centre
            closest_idx = int(round(bin_centre))
            closest_idx = min(closest_idx, n_total - 1)  # clamp to valid range
            selected_indices.append(closest_idx)

        seeds = group_sorted.iloc[selected_indices].copy()
        seeds['seed_index'] = range(1, n_seeds + 1)  # label which seed (1-8)
        seed_frames.append(seeds)

        print(f"  {library}: selected areas {sorted(seeds['area'].tolist())}")

    return pd.concat(seed_frames, ignore_index=True)


# ─────────────────────────────────────────────────────────────────────────────
# SPLIT INTO SEED + POOL
# ─────────────────────────────────────────────────────────────────────────────

def split_seed_pool(
    df      : pd.DataFrame,
    seeds   : pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the unified dataset into:
    - seed_set : the 24 points given to the GP at the start
    - pool_set : all remaining points — the GP predicts these,
                 and we reveal them one-by-one during active learning

    Parameters
    ----------
    df    : full unified dataset (966 rows)
    seeds : selected seed points (24 rows)

    Returns
    -------
    (seed_set, pool_set) as DataFrames
    """
    # Create a set of (library, area) tuples for fast lookup
    seed_keys = set(zip(seeds['library'], seeds['area']))

    # Tag each row in the full dataset
    df['is_seed'] = df.apply(
        lambda row: (row['library'], row['area']) in seed_keys,
        axis=1
    )

    seed_set = df[df['is_seed']].drop(columns=['is_seed']).reset_index(drop=True)
    pool_set = df[~df['is_seed']].drop(columns=['is_seed']).reset_index(drop=True)

    return seed_set, pool_set

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def run_seed_selection(
    unified_path : str,
    n_seeds      : int = 8,
    output_dir   : str = "data",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load unified data, select seeds, split into seed/pool, save both.

    Parameters
    ----------
    unified_path : path to unified_data.csv (output of Module 1)
    n_seeds      : number of seed points per library (default 8)
    output_dir   : folder to save seed_set.csv and pool_set.csv

    Returns
    -------
    (seed_set, pool_set)
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # ── Load ──────────────────────────────────────────────────────────────────
    print("=" * 55)
    print("STEP 1: Loading unified dataset")
    print("=" * 55)
    df = pd.read_csv(unified_path)
    print(f"  Loaded {len(df)} rows from {unified_path}")

    # ── Select seeds ──────────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print(f"STEP 2: Selecting {n_seeds} seed points per library")
    print("=" * 55)
    seeds = select_seeds_per_library(df, n_seeds=n_seeds)

    # ── Split ─────────────────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("STEP 3: Splitting into seed set and pool set")
    print("=" * 55)
    seed_set, pool_set = split_seed_pool(df, seeds)

    # ── Save ──────────────────────────────────────────────────────────────────
    seed_path = output_dir / "seed_set.csv"
    pool_path = output_dir / "pool_set.csv"

    seed_set.to_csv(seed_path, index=False)
    pool_set.to_csv(pool_path, index=False)

    # ── Summary ───────────────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("DONE — SUMMARY")
    print("=" * 55)
    print(f"  Seed set : {len(seed_set)} points → {seed_path}")
    print(f"  Pool set : {len(pool_set)} points → {pool_path}")
    print()
    print("Seed set composition coverage:")
    print(seed_set[['library', 'area', 'Au [at.%]', 'Ir [at.%]', 'log_k0', 'alpha']].to_string(index=False))

    return seed_set, pool_set


if __name__ == "__main__":

    print("\n" + "=" * 55)
    print("MODULE 2 — SEED SELECTION")
    print("=" * 55)

    # ── User inputs ───────────────────────────────────────────────────────────
    unified_path = input("\nPath to unified_data.csv:\n> ").strip().strip('"').strip("'")

    n_seeds_input = input("\nNumber of seed points per library [default: 8]:\n> ").strip()
    n_seeds = int(n_seeds_input) if n_seeds_input else 8

    output_dir = input("\nFolder to save seed_set.csv and pool_set.csv [default: data]:\n> ").strip().strip('"').strip("'")
    output_dir = output_dir if output_dir else "data"

    # ── Run ───────────────────────────────────────────────────────────────────
    seed_set, pool_set = run_seed_selection(
        unified_path = unified_path,
        n_seeds      = n_seeds,
        output_dir   = output_dir,
    )


MODULE 2 — SEED SELECTION



Path to unified_data.csv:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data\unified_data.csv

Number of seed points per library [default: 8]:
>  

Folder to save seed_set.csv and pool_set.csv [default: data]:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data


STEP 1: Loading unified dataset
  Loaded 966 rows from C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data\unified_data.csv

STEP 2: Selecting 8 seed points per library
  Au-rich: selected areas [22, 71, 112, 152, 192, 232, 273, 322]
  Ir-rich: selected areas [22, 71, 112, 152, 192, 232, 273, 322]
  Rh-rich: selected areas [22, 71, 112, 152, 192, 232, 273, 322]

STEP 3: Splitting into seed set and pool set

DONE — SUMMARY
  Seed set : 24 points → C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data\seed_set.csv
  Pool set : 942 points → C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data\pool_set.csv

Seed set composition coverage:
library  area  Au [at.%]  Ir [at.%]    log_k0    alpha
Au-rich    22      40.31      41.70 -4.513112 0.319202
Au-rich    71      46.82      31.86 -4.531695 0.322305
Au-rich   112      48.78      25.00 -4.500238 0.319490
Au-ric